## Standard Errors for κ (Kappa)

The standard errors for κ reported in stage 2 are currently not reliable.  
The reason is that stage 2 treats the predictions generated in stage 1 as fixed regressors, even though these predictions are themselves estimated quantities.

This creates a **generated-regressor problem**: the uncertainty from the stage 1 estimation is ignored in stage 2. As a result, the reported standard errors for κ are downward biased and do not correctly reflect the true sampling variability.

To address this issue, we bootstrap the **entire two-stage estimation procedure** rather than treating stage 2 in isolation. In each bootstrap replication, the full workflow is re-estimated: the stage 1 model is refit, new predictions are generated, and stage 2 is re-estimated using these predictions.

This approach propagates the estimation uncertainty from stage 1 into stage 2, thereby accounting for the generated-regressor problem. The resulting bootstrap distribution of κ allows us to construct standard errors and confidence intervals that correctly reflect uncertainty from both stages of the estimation process.


### Moving Block Bootstrap

Classic bootstrapping means:

1. Sample a new dataset by drawing from the training data with replacement until it has the same size as the original dataset.
2. Repeat this many times until you have enough synthetic datasets.
3. Estimate the model on all synthetic datasets.
4. Approximate the distribution of your point estimate by using the variation in point estimates generated on the synthetic data.

Basically, we change the underlying data a bit to see how our estimate would change.
The problem with timeseries data is, that we cannot simply draw observations with replacement becasue adjacent observations/time points are autocorrelated.

In Moving Block Bootstrap (MBB) draws are done over the ‘blocked sets’ of consecutive data, instead of over individual data rows.

In [1]:
import pandas as pd
import numpy as np
import random
from bootstrap import bootstrap_two_stage_block
from grid_search import estimate_single_config

c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [3]:
# set seed
random.seed(42)

# select a response variable (either marekt return or sp500 return)
y = response_variables['sprtrn']  # or 'sprtrn' for SP500 returns or vwretx

# transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# User-selected number of each 
num_topics = len(topic_cols)  
num_stocks = 0

# Randomly sample (without replacement), limited by available count
selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# Final filtered dataframe
X = X[selected_topics + selected_stocks]

# Convert stock returns to log returns: log(1+r)
X[selected_stocks] = np.log(X[selected_stocks] + 1)

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [13]:
result = bootstrap_two_stage_block(
    X=X, y=y,
    window_size=380, n_lags=15, lambda_val=0.0024140828549748896,
    B=100,
    block_len=20,          
    standardize=True,
    random_state=123
)

Block bootstrap (2-stage): 100%|██████████| 100/100 [1:25:26<00:00, 51.27s/it]


In [9]:
print(result["bootstrap_summary"])
print(result["draws"][["kappa", "intercept", "r2_insample_stage2", "r2_oos_stage2"]].describe())

{'B_requested': 100, 'B_success': 100, 'failure_rate': 0.0, 'block_len': 22, 'kappa_point': np.float64(0.663503959427171), 'kappa_boot_se': 0.35671451383974423, 'kappa_ci_2p5_97p5': (np.float64(4.027814544238083e-09), np.float64(0.9553131670322415)), 'intercept_point': np.float64(0.00041463191571027426), 'intercept_boot_se': 0.0001682058244457465, 'intercept_ci_2p5_97p5': (np.float64(9.040887655592385e-05), np.float64(0.0006978747340266843))}
              kappa   intercept  r2_insample_stage2  r2_oos_stage2
count  1.000000e+02  100.000000        1.000000e+02     100.000000
mean   2.970528e-01    0.000407        5.015986e-04      -0.222091
std    3.567145e-01    0.000168        1.605485e-03       2.108572
min    9.330870e-14    0.000033       -1.241485e-07     -21.090176
25%    4.740912e-06    0.000288       -1.969182e-09      -0.005342
50%    1.210073e-03    0.000401       -1.269096e-12      -0.001984
75%    5.806010e-01    0.000527        1.793273e-04      -0.000254
max    9.938999e-

In [10]:
res  = estimate_single_config(
    X, y,
    window_size= 380,
    n_lags=15,
    lambda_val=0.002376786911677026,
    standardize=True,   
    verbose=True,
    return_details=True
)

In [11]:
print(res)

{'summary': {'window_size': 380, 'n_lags': 15, 'lambda': 0.002376786911677026, 'r2_insample_stage1': np.float64(-0.0006315938217396775), 'r2_oos_stage1': np.float64(-0.0018089948992334737), 'r2_insample_stage2': np.float64(0.007352094116623076), 'r2_oos_stage2': np.float64(-0.00021389599861243447), 'kappa': np.float64(0.9545324959859449), 'kappa_tstat': np.float64(73.8042638733625), 'intercept': np.float64(0.00042124796441678485), 'intercept_tstat': np.float64(1.7502012249470607), 'n_observations': 1490, 'n_windows': 1492, 'n_oos_predictions_stage2': nan}, 'details':            date  window_size  n_lags    lambda  window_index  lasso_intercept  \
0    2011-07-27          380      15  0.002377             0              0.0   
1    2011-07-28          380      15  0.002377             1              0.0   
2    2011-07-29          380      15  0.002377             2              0.0   
3    2011-08-01          380      15  0.002377             3              0.0   
4    2011-08-02      